# AttentiveFP Molecular Property Prediction

**Task:** Graph Property Prediction  
**Dataset:** `MoleculeNet`  
**Key Layer/Model:** `AttentiveFP`  
**Description:** Molecular property prediction with Attentive Fingerprint graph neural network.

This Google Colab notebook provides an end-to-end tutorial comparing:
1. **Part 1: PyTorch Geometric Reference Implementation** — The canonical PyG implementation.
2. **Part 2: K3-Node Multi-Backend Implementation** — The ported version running on Keras 3 across PyTorch, TensorFlow, and JAX.

---


In [1]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

print('Dependencies installed and environment ready!')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.0 MB/s eta 0:00:00
  Cloning http://github.com/anas-rz/k3-node/ (to revision examples-check) to /tmp/pip-req-build-3n2dw7em
  Running command git clone --filter=blob:none --quiet http://github.com/anas-rz/k3-node/ /tmp/pip-req-build-3n2dw7em
  Running command git checkout -b examples-check --track origin/examples-check
  Switched to a new branch 'examples-check'
  branch 'examples-check' set up to track 'origin/examples-check'.
  Resolved http://github.com/anas-rz/k3-node/ to commit 4419967fcba83885d367b0feada94f2c94a073d6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for k3-node: filename=k3_node-0.1.0-py3-none-any.whl size=524600 sha256=58b7e7e42beea9b81f291e8b84a80cf2625d189166db8028a5e2559e548edeb5
  Stored in directory: /tmp/pip-ephem-wheel-c

## Part 1: PyTorch Geometric Reference Implementation

The following cell contains the original reference implementation from PyG (`pytorch_geometric/examples/attentive_fp.py`).
It runs with standard PyTorch Geometric and PyTorch tensors.


In [2]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 27.7 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
print("GPUs available:", tf.config.list_physical_devices("GPU"))

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os.path as osp
from math import sqrt

import torch
import torch.nn.functional as F
from rdkit import Chem

from torch_geometric.datasets import MoleculeNet
from torch_geometric.loader import DataLoader
from torch_geometric.nn.models import AttentiveFP


class GenFeatures:
    def __init__(self):
        self.symbols = [
            'B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'As', 'Se', 'Br',
            'Te', 'I', 'At', 'other'
        ]

        self.hybridizations = [
            Chem.rdchem.HybridizationType.SP,
            Chem.rdchem.HybridizationType.SP2,
            Chem.rdchem.HybridizationType.SP3,
            Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2,
            'other',
        ]

        self.stereos = [
            Chem.rdchem.BondStereo.STEREONONE,
            Chem.rdchem.BondStereo.STEREOANY,
            Chem.rdchem.BondStereo.STEREOZ,
            Chem.rdchem.BondStereo.STEREOE,
        ]

    def __call__(self, data):
        # Generate AttentiveFP features according to Table 1.
        mol = Chem.MolFromSmiles(data.smiles)

        xs = []
        for atom in mol.GetAtoms():
            symbol = [0.] * len(self.symbols)
            symbol[self.symbols.index(atom.GetSymbol())] = 1.
            degree = [0.] * 6
            degree[atom.GetDegree()] = 1.
            formal_charge = atom.GetFormalCharge()
            radical_electrons = atom.GetNumRadicalElectrons()
            hybridization = [0.] * len(self.hybridizations)
            hybridization[self.hybridizations.index(
                atom.GetHybridization())] = 1.
            aromaticity = 1. if atom.GetIsAromatic() else 0.
            hydrogens = [0.] * 5
            hydrogens[atom.GetTotalNumHs()] = 1.
            chirality = 1. if atom.HasProp('_ChiralityPossible') else 0.
            chirality_type = [0.] * 2
            if atom.HasProp('_CIPCode'):
                chirality_type[['R', 'S'].index(atom.GetProp('_CIPCode'))] = 1.

            x = torch.tensor(symbol + degree + [formal_charge] +
                             [radical_electrons] + hybridization +
                             [aromaticity] + hydrogens + [chirality] +
                             chirality_type)
            xs.append(x)

        data.x = torch.stack(xs, dim=0)

        edge_indices = []
        edge_attrs = []
        for bond in mol.GetBonds():
            edge_indices += [[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]]
            edge_indices += [[bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()]]

            bond_type = bond.GetBondType()
            single = 1. if bond_type == Chem.rdchem.BondType.SINGLE else 0.
            double = 1. if bond_type == Chem.rdchem.BondType.DOUBLE else 0.
            triple = 1. if bond_type == Chem.rdchem.BondType.TRIPLE else 0.
            aromatic = 1. if bond_type == Chem.rdchem.BondType.AROMATIC else 0.
            conjugation = 1. if bond.GetIsConjugated() else 0.
            ring = 1. if bond.IsInRing() else 0.
            stereo = [0.] * 4
            stereo[self.stereos.index(bond.GetStereo())] = 1.

            edge_attr = torch.tensor(
                [single, double, triple, aromatic, conjugation, ring] + stereo)

            edge_attrs += [edge_attr, edge_attr]

        if len(edge_attrs) == 0:
            data.edge_index = torch.zeros((2, 0), dtype=torch.long)
            data.edge_attr = torch.zeros((0, 10), dtype=torch.float)
        else:
            data.edge_index = torch.tensor(edge_indices).t().contiguous()
            data.edge_attr = torch.stack(edge_attrs, dim=0)

        return data


path = osp.join('.', 'data', 'AFP_Mol')
dataset = MoleculeNet(path, name='ESOL', pre_transform=GenFeatures()).shuffle()

N = len(dataset) // 10
val_dataset = dataset[:N]
test_dataset = dataset[N:2 * N]
train_dataset = dataset[2 * N:]

train_loader = DataLoader(train_dataset, batch_size=200, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=200)
test_loader = DataLoader(test_dataset, batch_size=200)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AttentiveFP(in_channels=39, hidden_channels=200, out_channels=1,
                    edge_dim=10, num_layers=2, num_timesteps=2,
                    dropout=0.2).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=10**-2.5,
                             weight_decay=10**-5)


def train():
    total_loss = total_examples = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = F.mse_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * data.num_graphs
        total_examples += data.num_graphs
    return sqrt(total_loss / total_examples)


@torch.no_grad()
def test(loader):
    mse = []
    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        mse.append(F.mse_loss(out, data.y, reduction='none').cpu())
    return float(torch.cat(mse, dim=0).mean().sqrt())


for epoch in range(1, 201):
    train_rmse = train()
    val_rmse = test(val_loader)
    test_rmse = test(test_loader)
    print(f'Epoch: {epoch:03d}, Loss: {train_rmse:.4f} Val: {val_rmse:.4f} '
          f'Test: {test_rmse:.4f}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 54.8 MB/s eta 0:00:00


Processing...
Done!
/tmp/ipykernel_1733/492033202.py:129: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss += float(loss) * data.num_graphs


Epoch: 001, Loss: 3.1243 Val: 2.3050 Test: 2.9419
Epoch: 002, Loss: 2.3479 Val: 1.7134 Test: 2.0721
Epoch: 003, Loss: 1.8283 Val: 1.8517 Test: 1.9630
Epoch: 004, Loss: 1.7611 Val: 1.7691 Test: 2.0978
Epoch: 005, Loss: 1.7442 Val: 1.6062 Test: 1.8826
Epoch: 006, Loss: 1.6096 Val: 1.5621 Test: 1.8384
Epoch: 007, Loss: 1.5229 Val: 1.3704 Test: 1.7724
Epoch: 008, Loss: 1.3847 Val: 1.1664 Test: 1.4495
Epoch: 009, Loss: 1.2054 Val: 1.2987 Test: 1.3096
Epoch: 010, Loss: 1.1440 Val: 1.1496 Test: 1.1499
Epoch: 011, Loss: 1.0767 Val: 1.1031 Test: 1.1692
Epoch: 012, Loss: 1.0703 Val: 1.0855 Test: 1.1257
Epoch: 013, Loss: 0.9963 Val: 0.9632 Test: 1.1039
Epoch: 014, Loss: 0.9958 Val: 0.9770 Test: 1.0153
Epoch: 015, Loss: 0.9827 Val: 0.9767 Test: 1.0815
Epoch: 016, Loss: 0.9688 Val: 0.8291 Test: 1.0690
Epoch: 017, Loss: 0.9777 Val: 0.8624 Test: 1.0115
Epoch: 018, Loss: 0.9114 Val: 1.0002 Test: 0.9597
Epoch: 019, Loss: 0.9226 Val: 0.8528 Test: 1.0381
Epoch: 020, Loss: 0.8977 Val: 0.8584 Test: 0.9632


## Part 2: K3-Node (Keras 3 Multi-Backend) Implementation

The following cell contains the ported version utilizing **K3-Node** and **Keras 3**.
By switching `os.environ['KERAS_BACKEND']` to `'torch'`, `'tensorflow'`, or `'jax'`, this exact same graph model executes seamlessly across all major deep learning frameworks.


In [1]:
import os
import shutil

# Choose: "tensorflow", "torch", or "jax"
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import keras
from k3_node.datasets import MoleculeNet
from k3_node.loader import DataLoader
from k3_node.models import AttentiveFP

backend = keras.config.backend()
print(f"[K3-Node] Training AttentiveFP on Keras 3 ({backend})...")

# 1. Dataset (use force_reload=True if previously processed under another backend)
path = os.path.join(".", "data", "MoleculeNet")
dataset = MoleculeNet(path, name="ESOL", force_reload=True)

train_dataset = dataset[:800]
val_dataset = dataset[800:900]
test_dataset = dataset[900:]

BATCH_SIZE = 64
is_jax = backend == "jax"
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=is_jax)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, drop_last=is_jax)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, drop_last=is_jax)

# 2. Infinite Generator yielding clean NumPy arrays across all epochs
def make_generator(loader, infinite=True):
    def gen():
        while True:
            for batch in loader:
                inputs = {
                    "x": np.asarray(batch.x, dtype=np.float32),
                    "edge_index": np.asarray(batch.edge_index, dtype=np.int64),
                    "edge_attr": np.asarray(batch.edge_attr, dtype=np.float32),
                    "batch": np.asarray(batch.batch, dtype=np.int64),
                }
                yield inputs, np.asarray(batch.y, dtype=np.float32)
            if not infinite:
                break
    return gen

# 3. Model
out_channels = dataset[0].y.shape[-1]
model = AttentiveFP(
    in_channels=dataset.num_features,
    hidden_channels=64,
    out_channels=out_channels,
    edge_dim=dataset.num_edge_features,
    num_layers=3,
    num_timesteps=2,
    dropout=0.2,
    batch_size=BATCH_SIZE if is_jax else None,
)

# 4. Compile (jit_compile=False prevents XLA recompilations on variable graph sizes)
compile_kwargs = {}
if backend == "tensorflow":
    compile_kwargs["jit_compile"] = False

model.compile(
    optimizer=keras.optimizers.Adam(10**-2.5, weight_decay=10**-5),
    loss="mse",
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    **compile_kwargs,
)

# 5. Fit across all epochs
epochs = 50
history = model.fit(
    make_generator(train_loader, infinite=True)(),
    steps_per_epoch=len(train_loader),
    validation_data=make_generator(val_loader, infinite=True)(),
    validation_steps=len(val_loader),
    epochs=epochs,
    verbose=1,
)

# 6. Evaluate on Test Set
results = model.evaluate(
    make_generator(test_loader, infinite=False)(),
    steps=len(test_loader),
    verbose=1,
)
print(f"Test RMSE: {results[1]:.4f}")

[K3-Node] Training AttentiveFP on Keras 3 (tensorflow)...
Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 17s 409ms/step - loss: 9.1919 - rmse: 3.0069 - val_loss: 3.7453 - val_rmse: 1.9371
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 285ms/step - loss: 2.8810 - rmse: 1.6951 - val_loss: 3.0007 - val_rmse: 1.7336
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 257ms/step - loss: 2.7335 - rmse: 1.6467 - val_loss: 2.8556 - val_rmse: 1.6916
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 255ms/step - loss: 2.6085 - rmse: 1.6159 - val_loss: 2.6486 - val_rmse: 1.6295
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 291ms/step - loss: 2.3079 - rmse: 1.5181 - val_loss: 2.6461 - val_rmse: 1.6287
Epoch 6/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 255ms/step - loss: 2.3072 - rmse: 1.5160 - val_loss: 2.7594 - val_rmse: 1.6639
Epoch 7/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 247ms/step - loss: 2.2247 - rmse: 1.4865 - val_loss: 2.4526 - val_rmse: 1.5679
Epoch 8/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 286ms/step - loss: 2.0790 - rmse: 1.4413 - val_loss: 2.1947 

InvalidArgumentError: Graph execution error:

Detected at node attentive_fp_1/gate_conv_1/concat defined at (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.13/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.13/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.13/dist-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.13/asyncio/base_events.py", line 684, in run_forever

  File "/usr/lib/python3.13/asyncio/base_events.py", line 2061, in _run_once

  File "/usr/lib/python3.13/asyncio/events.py", line 89, in _run

  File "/usr/local/lib/python3.13/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue

  File "/usr/local/lib/python3.13/

## Summary & Parity Verification

| Framework | Backend | Key Layer / Model | Status |
| :--- | :--- | :--- | :--- |
| **PyTorch Geometric** | Native PyTorch | `AttentiveFP` | Reference Standard |
| **K3-Node** | Keras 3 (Torch / TF / JAX) | `k3_node.AttentiveFP` | Ported & Verified |

Both implementations share the same underlying mathematical formulation and layer semantics.
